<a href="https://colab.research.google.com/github/deepshresthaa/A-Clustered-Graph-Based-Framework-for-Semantic-Research-Paper-Retrieval-and-Recommendation/blob/main/code/06_testing_the_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas==2.2.3
!pip install torch-geometric
!pip install sentence-transformers
!pip install -q --extra-index-url=https://pypi.nvidia.com "cudf-cu12" "cuml-cu12"
# Install pandas==2.2.3 to satisfy google-colab's requirements.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.3 which is incompatible.


In [ ]:
import pickle
from sentence_transformers import SentenceTransformer

print("Loading Sentence-Transformer...")
embed_model = SentenceTransformer("all-mpnet-base-v2")




Loading Sentence-Transformer...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [ ]:
print("Embedding formatted abstract...")
raw_embedding = embed_model.encode([academic_abstract])  # Shape: (1, 768)

In [ ]:
import numpy as np

# Load back the arrays instantly anywhere
loaded = np.load("/content/drive/MyDrive/pca_model_weights.npz")
pca_components = loaded["components"]
pca_mean = loaded["mean"]

print("Loaded components shape:", pca_components.shape)  # (100, 768)
print("Loaded mean shape:", pca_mean.shape)              # (768,)

# Transform any new embedding vector X_new:
# X_reduced = (X_new - pca_mean) @ pca_components.T

Loaded components shape: (100, 768)
Loaded mean shape: (768,)


In [ ]:
import numpy as np

class PCAPresenter:
    def __init__(self, npz_path):
        data = np.load(npz_path)
        self.mean_ = data["mean"]
        self.components_ = data["components"]

    def transform(self, X):
        # Ensure X is a numpy array
        X = np.asarray(X)
        # Apply the exact PCA linear transformation formula
        return (X - self.mean_) @ self.components_.T

# 2. Load it into the 'pca' variable
pca = PCAPresenter("/content/drive/MyDrive/pca_model_weights.npz")
print("Loaded successfully into the 'pca' variable!")


Loaded successfully into the 'pca' variable!


In [ ]:
import os
import torch
import numpy as np
import networkx as nx

class ClusterPredictor:
    def __init__(self, output_dir):
        self.output_dir = output_dir

        # Load the pre-computed centroids file directly (Zero looping over 45 files!)
        centroids_path = os.path.join(self.output_dir, "cluster_centroids.pt")
        if not os.path.exists(centroids_path):
            raise FileNotFoundError(f"Centroids file not found at {centroids_path}. Run the pre-computation step first!")

        print("Loading pre-computed cluster centroids from storage...")
        saved_data = torch.load(centroids_path, map_location='cpu', weights_only=False)

        self.cluster_ids = saved_data["cluster_ids"]
        self.cluster_centers_dict = saved_data["cluster_centers_dict"]
        self.cluster_centers_matrix = saved_data["cluster_centers_matrix"]
        print(f"Successfully initialized with {len(self.cluster_ids)} cached clusters.")

    def predict(self, query_vector, p=0.7442, q=2.9471, max_depth=3, threshold=0.15):
        # Step 1: Fast centroid routing
        query_norm = query_vector / np.linalg.norm(query_vector)
        centers_norm = self.cluster_centers_matrix / np.linalg.norm(self.cluster_centers_matrix, axis=1, keepdims=True)
        cluster_sims = centers_norm @ query_norm

        best_idx = int(np.argmax(cluster_sims))
        target_cluster = self.cluster_ids[best_idx]
        print(f"Routing to Cluster ID: {target_cluster} (Centroid Cosine Similarity: {cluster_sims[best_idx]:.4f})")

        # Step 2: Load target cluster data
        checkpoint_path = os.path.join(self.output_dir, f"cluster_{target_cluster}.pt")
        cluster_data = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        graph_data = cluster_data['graph_data']

        # Step 3: Localized graph search
        query_t = torch.as_tensor(query_vector, dtype=torch.float32).unsqueeze(0)
        node_sims = torch.cosine_similarity(query_t, graph_data.x, dim=1).detach().numpy()
        seed_idx = int(np.argmax(node_sims))

        G = nx.Graph()
        edges = graph_data.edge_index.numpy()
        for u, v in zip(edges[0], edges[1]):
            G.add_edge(u, v)

        visited_scores = {}
        queue = [(seed_idx, 0, 1.0)]

        while queue:
            curr_node, depth, prob = queue.pop(0)
            if curr_node in visited_scores or depth > max_depth:
                continue

            combined_score = 0.6 * float(node_sims[curr_node]) + 0.4 * prob
            visited_scores[curr_node] = combined_score

            if depth < max_depth:
                neighbors = list(G.neighbors(curr_node)) if G.has_node(curr_node) else []
                for nbr in neighbors:
                    bias = (1 / p) if nbr == seed_idx else (1 / q)
                    if len(neighbors) > 0:
                        next_prob = prob * (1.0 / len(neighbors)) * bias
                        queue.append((nbr, depth + 1, next_prob))

        # Step 4: Compile results with adjusted threshold filtering
        results = []
        for node_idx, score in visited_scores.items():
            if score >= threshold:
                results.append({
                    "paper_id": graph_data.paper_ids[node_idx],
                    "title": graph_data.titles[node_idx],
                    "score": round(score, 4)
                })

        return target_cluster, sorted(results, key=lambda x: x["score"], reverse=True)

# ==========================================
# Example usage in an independent script:
# ==========================================
if __name__ == "__main__":
    from google.colab import drive
    drive.mount('/content/drive')

    OUTPUT_DIR = "/content/drive/MyDrive/gnn_cluster_models"

    # Initialize predictor once
    predictor = ClusterPredictor(OUTPUT_DIR)

    dummy_query = np.random.randn(predictor.cluster_centers_matrix.shape[1])
    cluster_id, recommendations = predictor.predict(dummy_query)

    print(f"\nTop Recommended Papers from Cluster {cluster_id}:")
    for r in recommendations[:5]:
        # Clean the paper_id to form a valid arXiv URL (e.g., convert 'abs-2512.13644v2' -> '2512.13644v2')
        raw_id = r['paper_id']
        clean_id = raw_id.replace("abs-", "")
        arxiv_url = f"https://arxiv.org/abs/{clean_id}"

        print(f" - [{r['score']}] {r['title']} ([Link]({arxiv_url}))")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading pre-computed cluster centroids from storage...
Successfully initialized with 45 cached clusters.
Routing to Cluster ID: 24 (Centroid Cosine Similarity: 0.2547)

Top Recommended Papers from Cluster 24:
 - [0.6512] Learning image-to-image translation using paired and unpaired training
  samples ([Link](https://arxiv.org/abs/1805.03189v1))
 - [0.3398] Unsupervised Multi-Domain Multimodal Image-to-Image Translation with
  Explicit Domain-Constrained Disentanglement ([Link](https://arxiv.org/abs/1911.00622v1))
 - [0.2111] Semantic Example Guided Image-to-Image Translation ([Link](https://arxiv.org/abs/1909.13028v2))
 - [0.2065] Exemplar Guided Unsupervised Image-to-Image Translation with Semantic
  Consistency ([Link](https://arxiv.org/abs/1805.11145v4))
 - [0.1932] Multimodal Unsupervised Image-to-Image Translation ([Link](https://arxiv.org/abs/1804.04732

## Using `Qwen/Qwen2.5-1.5B-Instruct` to rewrite the informal user query into a formal academic abstract paragraph suitable for a computer science research paper.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
import torch
import numpy as np

# -------------------------------------------------------------
# STEP 1: Decoder-Only Model Setup (for rewriting raw queries)
# -------------------------------------------------------------
print("Loading decoder model for query formatting...")
decoder_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(decoder_model_id)
decoder_llm = AutoModelForCausalLM.from_pretrained(
    decoder_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)



Loading decoder model for query formatting...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
def format_query_to_abstract(raw_user_query):
    messages = [
        {"role": "system", "content": "You are an AI research assistant. Rewrite the following informal user query into a formal academic abstract paragraph suitable for a computer science research paper. Focus on the core methodology, problem statement, and technical domain. Return ONLY the abstract text."},
        {"role": "user", "content": raw_user_query}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to(decoder_llm.device)

    generated_ids = decoder_llm.generate(
        **model_inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    formatted_abstract = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return formatted_abstract.strip()

# -------------------------------------------------------------
# STEP 2: Raw User Query Input
# -------------------------------------------------------------
raw_query = "how do robots learn to pick up things from watching people on youtube videos"
print(f"\n[Raw User Query]: {raw_query}")

# Format via Decoder Model
academic_abstract = format_query_to_abstract(raw_query)
print(f"\n[Decoder Formatted Abstract]:\n{academic_abstract}\n")

# -------------------------------------------------------------
# STEP 3: Embed, Load Saved PCA, and Reduce
# -------------------------------------------------------------
print("Loading Sentence-Transformer...")
embed_model = SentenceTransformer('all-mpnet-base-v2')

print("Embedding formatted abstract...")
raw_embedding = embed_model.encode([academic_abstract]) # Shape: (1, 768)

# Load your trained PCA weights from Google Drive
print("Loading saved PCA model weights...")
loaded_pca = np.load("/content/drive/MyDrive/pca_model_weights.npz")
pca_mean = loaded_pca["mean"]               # Shape: (768,)
pca_components = loaded_pca["components"]   # Shape: (100, 768)

print("Applying trained PCA reduction to 100 dimensions...")
# Uses the exact formula of your trained model: (X - mean) @ components.T
query_vector_100d = ((raw_embedding - pca_mean) @ pca_components.T)[0] # Shape: (100,)

# Run prediction via your ClusterPredictor class
cluster_id, recommendations = predictor.predict(query_vector_100d, threshold=0.15)
# -------------------------------------------------------------
# STEP 4: Display Results
# -------------------------------------------------------------
print(f"\nTop Recommended Papers from Cluster {cluster_id}:")
for r in recommendations[:5]:
    raw_id = r['paper_id']
    clean_id = raw_id.replace("abs-", "")
    arxiv_url = f"https://arxiv.org/abs/{clean_id}"

    print(f" - [{r['score']}] {r['title']} ([Link]({arxiv_url}))")


[Raw User Query]: how do robots learn to pick up things from watching people on youtube videos

[Decoder Formatted Abstract]:
The objective of this study is to investigate how robots can be trained to perform tasks such as picking up objects by mimicking human actions observed in YouTube videos. The methodology involves developing algorithms that enable robots to analyze and replicate human movements through video-based learning techniques. This approach aims to enhance robotic autonomy in real-world applications where direct interaction with humans may not always be feasible or desirable. The primary focus lies in understanding the transferability of learned skills between different types of tasks and environments, thereby contributing to advancements in robotics technology.

Loading Sentence-Transformer...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding formatted abstract...
Loading saved PCA model weights...
Applying trained PCA reduction to 100 dimensions...
Routing to Cluster ID: 1 (Centroid Cosine Similarity: 0.6934)

Top Recommended Papers from Cluster 1:
 - [0.9024] Robot Learning from Human Videos: A Survey ([Link](https://arxiv.org/abs/2604.27621v1))
 - [0.4826] Imitation from Observation: Learning to Imitate Behaviors from Raw Video
  via Context Translation ([Link](https://arxiv.org/abs/1707.03374v2))
 - [0.469] SafeMimic: Towards Safe and Autonomous Human-to-Robot Imitation for Mobile Manipulation ([Link](https://arxiv.org/abs/2506.15847v1))
 - [0.468] ZeroMimic: Distilling Robotic Manipulation Skills from Web Videos ([Link](https://arxiv.org/abs/2503.23877v1))
 - [0.4659] Imitating What Works: Simulation-Filtered Modular Policy Learning from Human Videos ([Link](https://arxiv.org/abs/2602.13197v2))
